# Multimodal Product Understanding (CLIP + FAISS + LLM)
Demo notebook for image-text retrieval and tagging.

In [ ]:
!pip install transformers torch faiss-cpu pillow openai pandas numpy

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import faiss
import numpy as np
import pandas as pd

In [ ]:
model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

In [ ]:
df = pd.read_csv('../data/sample_products.csv')
image_embeddings = []
for img_name in df['image']:
    image = Image.open(f'../images/{img_name}')
    inputs = processor(images=image, return_tensors='pt')
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
    image_embeddings.append(emb[0].numpy())

image_embeddings = np.array(image_embeddings)

In [ ]:
dimension = image_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(image_embeddings)
print('Index built:', index.ntotal)

In [ ]:
query = 'blue casual shirt'
inputs = processor(text=[query], return_tensors='pt', padding=True)
with torch.no_grad():
    query_embedding = model.get_text_features(**inputs)

query_embedding = query_embedding.numpy()
distances, indices = index.search(query_embedding, 3)

print('Top matches:')
for i in indices[0]:
    print(df.iloc[i]['description'])